# Catch the Coin Using Q-Learning

**AI Studio Project**  
**Student:** Your Name

This project demonstrates how a Q-Learning agent learns to catch a falling coin using reinforcement learning. The agent improves by playing many episodes and updating a Q-table based on rewards.

## 1. Import Required Libraries

The project uses simple Python libraries. `random` is used for random coin positions and exploration, `defaultdict` stores the Q-table, `time` slows down the demonstration, and `matplotlib` is used for visualizations.

In [ ]:
import random
import time
from collections import defaultdict
import matplotlib.pyplot as plt

## 2. Define Game Actions

The agent can choose one of three actions: move left, stay in the same place, or move right.

In [ ]:
LEFT = 0
STAY = 1
RIGHT = 2

## 3. Create the Game Environment

The `CoinGame` class represents the environment. The coin starts at the top of the grid and moves down one row after each action. The agent is located at the bottom row and tries to catch the coin.

In [ ]:
class CoinGame:
    def __init__(self, rows=5, cols=5):
        self.rows = rows
        self.cols = cols
        self.reset()

    def reset(self):
        # Start the agent in the middle column
        self.agent_col = self.cols // 2

        # Start the coin in a random column at the top row
        self.coin_col = random.randint(0, self.cols - 1)
        self.coin_row = 0

        self.done = False
        return self.get_state()

    def get_state(self):
        # State = agent position, coin column, coin row
        return (self.agent_col, self.coin_col, self.coin_row)

    def available_actions(self):
        # The agent can move left, stay, or move right
        return [LEFT, STAY, RIGHT]

    def step(self, action):
        # Move the agent based on the selected action
        if action == LEFT:
            self.agent_col -= 1
        elif action == RIGHT:
            self.agent_col += 1

        # Keep the agent inside the grid
        self.agent_col = max(0, min(self.agent_col, self.cols - 1))

        # Move the coin down by one row
        self.coin_row += 1

        reward = 0
        caught = False

        # If the coin reaches the bottom row, the episode ends
        if self.coin_row == self.rows - 1:
            self.done = True

            if self.agent_col == self.coin_col:
                reward = 10
                caught = True
            else:
                reward = -10

        return self.get_state(), reward, self.done, caught

    def render(self):
        # Print the current game grid in the terminal
        for row in range(self.rows):
            line = []
            for col in range(self.cols):
                if (
                    row == self.coin_row
                    and col == self.coin_col
                    and row == self.rows - 1
                    and col == self.agent_col
                ):
                    line.append("X")
                elif row == self.coin_row and col == self.coin_col:
                    line.append("C")
                elif row == self.rows - 1 and col == self.agent_col:
                    line.append("A")
                else:
                    line.append(".")
            print(" ".join(line))
        print()

## 4. Create the Q-Learning Agent

The `QLearningAgent` class stores learned values in a Q-table. The agent uses epsilon-greedy action selection, meaning it sometimes explores random actions and sometimes chooses the best known action.

In [ ]:
class QLearningAgent:
    def __init__(
        self,
        alpha=0.1,
        gamma=0.9,
        epsilon=1.0,
        epsilon_decay=0.995,
        epsilon_min=0.01
    ):
        self.q_table = defaultdict(float)

        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def get_q(self, state, action):
        return self.q_table[(state, action)]

    def choose_action(self, state, actions):
        # Exploration: choose a random action
        if random.random() < self.epsilon:
            return random.choice(actions)

        # Exploitation: choose the action with the highest Q-value
        q_values = [self.get_q(state, action) for action in actions]
        max_q = max(q_values)

        best_actions = []
        for action in actions:
            if self.get_q(state, action) == max_q:
                best_actions.append(action)

        return random.choice(best_actions)

    def update(self, state, action, reward, next_state, actions, done):
        # Update the Q-table using the Q-learning formula
        old_q = self.get_q(state, action)

        if done:
            target = reward
        else:
            next_max_q = max(
                self.get_q(next_state, next_action)
                for next_action in actions
            )
            target = reward + self.gamma * next_max_q

        new_q = old_q + self.alpha * (target - old_q)
        self.q_table[(state, action)] = new_q

    def decay_epsilon(self):
        # Reduce exploration gradually after each episode
        self.epsilon = max(
            self.epsilon_min,
            self.epsilon * self.epsilon_decay
        )

## 5. Moving Average Function

A moving average is used to smooth the graphs so that the training progress is easier to understand.

In [ ]:
def moving_average(values, window=100):
    averages = []

    for i in range(len(values)):
        start = max(0, i - window + 1)
        current_window = values[start:i + 1]
        averages.append(sum(current_window) / len(current_window))

    return averages

## 6. Train the Agent

The agent is trained over many episodes. In each episode, the coin starts again from the top and the agent tries to catch it. After each action, the Q-table is updated based on the reward received.

In [ ]:
def train(agent, episodes=5000):
    env = CoinGame()

    rewards = []
    catches = []

    for episode in range(episodes):
        state = env.reset()
        done = False
        total_reward = 0
        caught = False

        while not done:
            actions = env.available_actions()
            action = agent.choose_action(state, actions)

            next_state, reward, done, caught = env.step(action)

            agent.update(
                state,
                action,
                reward,
                next_state,
                actions,
                done
            )

            state = next_state
            total_reward += reward

        agent.decay_epsilon()

        rewards.append(total_reward)
        catches.append(1 if caught else 0)

        if (episode + 1) % 1000 == 0:
            recent_catch_rate = sum(catches[-1000:]) / 1000
            print(
                f"Episode {episode + 1}, "
                f"Catch rate: {recent_catch_rate * 100:.2f}%"
            )

    print()
    print("Training finished.")
    print("=" * 40)
    print("Training Summary")
    print("=" * 40)
    print("Episodes:", episodes)
    print("Final epsilon:", round(agent.epsilon, 4))
    print("Q-table size:", len(agent.q_table))
    print(
        "Final catch rate last 100 episodes:",
        round(sum(catches[-100:]) / 100 * 100, 2),
        "%"
    )
    print("=" * 40)

    return rewards, catches

## 7. Plot the Learning Results

Two graphs are created after training. The first graph shows the average reward over episodes. The second graph shows the catch rate over episodes.

In [ ]:
def plot_results(rewards, catches):
    reward_avg = moving_average(rewards)
    catch_avg = moving_average(catches)

    plt.figure(figsize=(10, 5))
    plt.plot(reward_avg)
    plt.title("Average Reward Over Episodes")
    plt.xlabel("Episode")
    plt.ylabel("Average Reward")
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(catch_avg)
    plt.title("Catch Rate Over Episodes")
    plt.xlabel("Episode")
    plt.ylabel("Catch Rate")
    plt.show()

## 8. Convert Actions to Text

This function converts the action numbers into readable words so that the demonstration is easier to follow.

In [ ]:
def action_name(action):
    if action == LEFT:
        return "Left"
    elif action == RIGHT:
        return "Right"
    else:
        return "Stay"

## 9. Watch the Trained Agent

After training, the agent plays test games using the learned policy. The exploration rate is set to zero, so the agent does not make random moves during testing.

In [ ]:
def watch_agent(agent, games=10):
    env = CoinGame()
    old_epsilon = agent.epsilon
    agent.epsilon = 0

    for game in range(games):
        print()
        print("Game", game + 1)
        print("-" * 30)

        state = env.reset()
        done = False
        caught = False

        env.render()

        while not done:
            action = agent.choose_action(state, env.available_actions())

            print("Agent action:", action_name(action))

            state, reward, done, caught = env.step(action)
            env.render()

            time.sleep(0.3)

        if caught:
            print("Result: Coin caught!")
        else:
            print("Result: Coin missed.")

    agent.epsilon = old_epsilon

## 10. Main Function

The main function creates the agent, trains it, plots the results, and then shows the trained agent playing the game.

In [ ]:
def main():
    print("=" * 40)
    print("Catch the Coin - Q-Learning Project")
    print("=" * 40)

    agent = QLearningAgent()

    rewards, catches = train(agent, episodes=5000)

    plot_results(rewards, catches)

    watch_agent(agent, games=10)

## 11. Run the Project

This cell starts the full project.

In [ ]:
if __name__ == "__main__":
    main()

# Conclusion

This project successfully implemented the Q-Learning algorithm in a simple game environment.

At the beginning of training, the agent selected many random actions because the exploration rate was high. Over time, the Q-table was updated and the agent learned which actions helped it catch the coin.

The training results show improvement in both average reward and catch rate. After 5000 episodes, the agent was able to catch the coin with a very high success rate. This demonstrates how reinforcement learning can help an agent learn from experience without being directly programmed with the correct actions.